<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-00-prerequisites/lesson-0.2-math-foundations/notebooks/exercises-0_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 0.2: Math Foundations for GenAI

*Module 0 · Prerequisites*

> You don't need a PhD. You need 9 concepts: vectors, cosine similarity, softmax, temperature, gradient descent, matrix multiplication (the glue of transformers), backpropagation (how gradients flow), perplexity (how to evaluate LLMs), and activation functions (why neural networks can learn anything). All in Python. No proofs.

`Prerequisite` · `9 Concepts` · `NumPy Code` · `90 min`

---

## About this notebook

This notebook contains the **7 runnable code examples** from the published lesson `lesson-0.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Vectors — How AI Sees Words — `vectors.py`
2. Step 2 — Dot Product & Cosine Similarity — "How Similar Are These?" — `similarity.py`
3. Step 3 — Softmax & Temperature — How AI Picks the Next Word — `softmax_temperature.py`
4. Step 4 — Gradient Descent — How Models Learn — `gradient_descent.py`
5. Step 5 — Matrix Multiplication — The Glue of Transformers — `matrix_multiply.py`
6. Step 6 — Backpropagation — How Gradients Actually Flow — `backpropagation.py`
7. Step 7 — Perplexity & Activation Functions — Evaluate + Power Up — `perplexity_activation.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers numpy


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Vectors — How AI Sees Words

To an LLM, "king" isn't letters — it's a list of 768 numbers that encode its meaning.

**`vectors.py`** · _Block 1 of 7_

VECTORS — How AI represents words — Each word becomes a list of numbers (embedding)


In [ ]:
# =============================================
# VECTORS — How AI represents words
# Each word becomes a list of numbers (embedding)
# that captures its MEANING.
# =============================================

import numpy as np

# Simplified word vectors (real ones have 768+ dimensions)
# Each dimension captures a property: royalty, gender, food, etc.
words = {
    "king":   np.array([0.9, 0.1, 0.8, 0.0]),  # royal, male
    "queen":  np.array([0.9, 0.9, 0.8, 0.0]),  # royal, female
    "man":    np.array([0.1, 0.1, 0.5, 0.0]),  # not royal, male
    "woman":  np.array([0.1, 0.9, 0.5, 0.0]),  # not royal, female
    "biryani": np.array([0.0, 0.0, 0.0, 0.9]),  # food
}

print("Word Vectors (simplified to 4 dimensions):\n")
for word, vec in words.items():
    print(f"  {word:8s} = {vec}")

# The famous equation: king - man + woman ≈ queen
result = words["king"] - words["man"] + words["woman"]
print(f"\n  king - man + woman = {result}")
print(f"  queen              = {words['queen']}")
print(f"  Match! Vectors capture MEANING, not just spelling.")


> **What just happened?** Each word became a vector (list of numbers) encoding its meaning. "King" and "queen" have similar vectors (both royal). "King" and "biryani" are completely different. The famous equation king - man + woman ≈ queen works because the vectors capture semantic relationships. This is how RAG works (Module 6): convert your documents into vectors, convert the user's question into a vector, find the closest match.


### Step 2 · Dot Product & Cosine Similarity — "How Similar Are These?"

The single most important operation in all of AI: measuring similarity between vectors.

**`similarity.py`** · _Block 2 of 7_

COSINE SIMILARITY — The heart of RAG — "How similar are two word vectors?"


In [ ]:
# =============================================
# COSINE SIMILARITY — The heart of RAG
# "How similar are two word vectors?"
# This is EXACTLY what happens when you search
# a vector database in Module 6.
# =============================================

import numpy as np

def cosine_similarity(a, b):
    """How similar are two vectors? Returns -1 to +1."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Simulate document embeddings (like a vector database)
documents = {
    "Python tutorial":    np.array([0.9, 0.8, 0.1, 0.0]),
    "JavaScript guide":   np.array([0.8, 0.7, 0.2, 0.0]),
    "Biryani recipe":     np.array([0.0, 0.1, 0.9, 0.8]),
    "Curry recipe":       np.array([0.0, 0.1, 0.8, 0.9]),
    "Machine learning":   np.array([0.7, 0.9, 0.3, 0.0]),
}

# User's query
query = np.array([0.85, 0.75, 0.15, 0.0])  # "How to code in Python"

print("RAG Retrieval Simulation:\n")
print(f"  Query: 'How to code in Python'\n")

results = []
for doc, vec in documents.items():
    sim = cosine_similarity(query, vec)
    results.append((doc, sim))
    bar = "█" * int(sim * 20)
    print(f"  {doc:20s} similarity={sim:.3f} {bar}")

best = max(results, key=lambda x: x[1])
print(f"\n  ✅ Best match: '{best[0]}' (similarity={best[1]:.3f})")
print(f"  This is EXACTLY how vector search works in ChromaDB/Pinecone.")


> **What just happened?** We simulated RAG retrieval: the query "How to code in Python" was converted to a vector, then compared against all document vectors using cosine similarity. "Python tutorial" scored highest (0.99) and was returned as the best match. "Biryani recipe" scored near zero — completely unrelated. This is the exact algorithm behind every vector database you'll use in Module 6.


### Step 3 · Softmax & Temperature — How AI Picks the Next Word

The model scores every possible next word. Softmax turns scores into probabilities. Temperature controls how "creative" the choice is.

**`softmax_temperature.py`** · _Block 3 of 7_

SOFTMAX + TEMPERATURE — Softmax: turn raw scores into probabilities.


In [ ]:
# =============================================
# SOFTMAX + TEMPERATURE
# Softmax: turn raw scores into probabilities.
# Temperature: control creativity.
# This runs inside every LLM for every token.
# =============================================

import numpy as np

def softmax(scores, temperature=1.0):
    """Turn raw scores into probabilities."""
    scaled = scores / temperature  # ← temperature divides the scores
    e = np.exp(scaled - np.max(scaled))  # subtract max for numerical stability
    return e / e.sum()

# The LLM scored 5 possible next words
words = ["the", "a", "his", "my", "that"]
raw_scores = np.array([9.2, 7.1, 5.8, 4.3, 3.0])

print("Next word after 'The cat sat on ___'\n")
print(f"  Raw scores: {dict(zip(words, raw_scores))}\n")

# Temperature comparison
for temp in [0.1, 0.5, 1.0, 2.0]:
    probs = softmax(raw_scores, temperature=temp)
    label = {0.1: "🧊 Frozen", 0.5: "📏 Focused",
             1.0: "⚖️ Balanced", 2.0: "🔥 Creative"}[temp]
    print(f"  temp={temp} {label}:")
    for w, p in zip(words, probs):
        bar = "█" * int(p * 40)
        print(f"    {w:5s} {p:.3f} {bar}")
    print()

print("Low temp → 'the' wins every time (predictable)")
print("High temp → any word could win (creative/random)")
print("This is the ONLY parameter behind 'creativity' in ChatGPT.")


> **What just happened?** At temp=0.1 : "the" gets 99.9% probability — the model ALWAYS picks the most likely word (deterministic, boring). At temp=2.0 : probabilities spread out — "a", "his", "my" all have meaningful chances (creative, surprising). Temperature doesn't change the MODEL — it changes how probabilities are distributed after softmax. Every GenAI API has a temperature parameter. Now you know exactly what it does mathematically.


### Step 4 · Gradient Descent — How Models Learn

The model makes a prediction. It's wrong. The gradient tells it HOW to adjust to be less wrong next time.

**`gradient_descent.py`** · _Block 4 of 7_

GRADIENT DESCENT — How ALL neural networks learn — Goal: find the value of x that minimizes f(x) = (x - 7)²


In [ ]:
# =============================================
# GRADIENT DESCENT — How ALL neural networks learn
# Goal: find the value of x that minimizes f(x) = (x - 7)²
# (Answer should be x = 7)
# =============================================

# The function to minimize (loss function)
def loss(x):
    return (x - 7) ** 2  # minimum at x=7

# The gradient (slope) — tells us which direction is "downhill"
def gradient(x):
    return 2 * (x - 7)  # derivative of (x-7)²

# Start with a random guess
x = 0.0
learning_rate = 0.1

print("Gradient Descent — finding the minimum:\n")
print(f"  Goal: find x that minimizes (x - 7)²")
print(f"  Starting at x = {x}\n")

for step in range(15):
    grad = gradient(x)
    x = x - learning_rate * grad  # THE update rule
    
    if step % 3 == 0:
        bar_pos = int(x * 3)
        bar = " " * max(0, bar_pos) + "●"
        print(f"  Step {step:2d}: x={x:.3f}  loss={loss(x):.4f}  [{bar}]")

print(f"\n  Final x = {x:.4f} (target was 7.0)")
print(f"  Final loss = {loss(x):.6f} (near zero = success!)")
print(f"\n  This is how GPT-4 learned from trillions of words:")
print(f"  predict next word → measure error → gradient → adjust weights → repeat")


> **What just happened?** Starting from x=0 (terrible guess), gradient descent took small steps toward x=7 (the correct answer). Each step: compute the gradient (which direction is downhill?), multiply by learning rate (how big a step?), update x. After 15 steps, x ≈ 7.0 and loss ≈ 0. This is EXACTLY how GPT trains: predict a word → compute loss (how wrong?) → compute gradient (which direction to adjust?) → update 175 billion parameters → repeat trillions of times.


### Step 5 · Matrix Multiplication — The Glue of Transformers

Every transformer layer is fundamentally a series of matrix multiplications. This is the operation that makes LLMs work.

**`matrix_multiply.py`** · _Block 5 of 7_

MATRIX MULTIPLICATION — The glue of transformers — Every transformer layer = a series of matmuls


In [ ]:
# =============================================
# MATRIX MULTIPLICATION — The glue of transformers
# Every transformer layer = a series of matmuls
# =============================================

import numpy as np

# ── A matrix TRANSFORMS vectors ──
# Input: 3 tokens, each with 4-dimensional embedding
tokens = np.array([
    [0.8, 0.1, 0.5, 0.2],  # "The"
    [0.2, 0.9, 0.3, 0.7],  # "cat"
    [0.1, 0.3, 0.8, 0.4],  # "sat"
])
print(f"Input tokens: {tokens.shape}")  # (3 tokens, 4 dims)

# ── Weight matrix: projects 4D → 3D ──
# In real transformers: 768D → 768D with 12 attention heads
W_query = np.random.randn(4, 3) * 0.1  # Learned during training
W_key   = np.random.randn(4, 3) * 0.1
W_value = np.random.randn(4, 3) * 0.1

# ── Matrix multiply: tokens × weights = Q, K, V ──
Q = tokens @ W_query   # "What am I looking for?"
K = tokens @ W_key     # "What do I contain?"
V = tokens @ W_value   # "What should I output?"

print(f"Q shape: {Q.shape}")  # (3, 3) — 3 tokens, 3-dim queries
print(f"K shape: {K.shape}")  # (3, 3) — 3 tokens, 3-dim keys

# ── Attention scores: Q × K^T ──
# How much should each token attend to every other token?
d_k = K.shape[1]
scores = (Q @ K.T) / np.sqrt(d_k)  # Scale by √d_k for stability

# ── Softmax → attention weights ──
def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attention = softmax(scores)
print(f"\nAttention weights (who looks at whom):")
print(f"  'The' attends to: The={attention[0,0]:.2f} cat={attention[0,1]:.2f} sat={attention[0,2]:.2f}")
print(f"  'cat' attends to: The={attention[1,0]:.2f} cat={attention[1,1]:.2f} sat={attention[1,2]:.2f}")

# ── Final output: attention × V ──
output = attention @ V
print(f"\nOutput shape: {output.shape}")  # (3, 3) — context-aware representations
print(f"Each token now contains information from tokens it attended to!")


> **What just happened?** Three matrix multiplications transformed our tokens into Q, K, V. Then QK T computed attention scores (which tokens should look at which). Softmax normalized them into weights. Then attention × V produced context-aware outputs. This is the complete self-attention mechanism — the core innovation of transformers. In Module 2, we'll implement multi-head attention (running this 12× in parallel with different weight matrices) and full transformer blocks.


### Step 6 · Backpropagation — How Gradients Actually Flow

Gradient descent tells us WHERE to go. Backpropagation tells us HOW to compute the gradients in a deep network.

**`backpropagation.py`** · _Block 6 of 7_

BACKPROPAGATION — How gradients flow backwards — The chain rule: ∂Loss/∂w = product of partial derivatives


In [ ]:
# =============================================
# BACKPROPAGATION — How gradients flow backwards
# The chain rule: ∂Loss/∂w = product of partial derivatives
# =============================================

import numpy as np

# ── Tiny neural network: 2 layers ──
# Input → Layer 1 (multiply by w1, add bias) → ReLU → Layer 2 (multiply by w2) → Output → Loss

np.random.seed(42)
x = np.array([2.0])           # Input
y_true = np.array([8.0])     # Target output
w1, w2 = 0.5, 0.3            # Weights to learn
lr = 0.01                     # Learning rate

print("Training a 2-layer network with backpropagation:\n")

for epoch in range(50):
    # ── FORWARD PASS (input → output) ──
    z1 = x * w1               # Layer 1: multiply
    a1 = np.maximum(0, z1)    # ReLU activation
    z2 = a1 * w2              # Layer 2: multiply
    loss = (z2 - y_true) ** 2 # MSE loss

    # ── BACKWARD PASS (chain rule: output → input) ──
    dL_dz2 = 2 * (z2 - y_true)          # ∂Loss/∂z2
    dL_dw2 = dL_dz2 * a1                # ∂Loss/∂w2 = ∂Loss/∂z2 × ∂z2/∂w2
    dL_da1 = dL_dz2 * w2                # ∂Loss/∂a1 = ∂Loss/∂z2 × ∂z2/∂a1
    dL_dz1 = dL_da1 * (z1 > 0)         # ReLU gradient: 1 if z1>0, else 0
    dL_dw1 = (dL_dz1 * x)[0]            # ∂Loss/∂w1 = chain all the way back

    # ── UPDATE WEIGHTS (gradient descent step) ──
    w1 -= lr * dL_dw1
    w2 -= lr * dL_dw2[0]

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:2d}: w1={w1:.4f} w2={w2:.4f} output={z2[0]:.4f} loss={loss[0]:.4f}")

print(f"\n  Final: input={x[0]} × w1={w1:.3f} × w2={w2:.3f} = {(x*w1*w2)[0]:.3f} (target: {y_true[0]})")
print(f"\n  In PyTorch, you just call loss.backward() — it computes ALL gradients automatically!")
print(f"  For GPT-4's 1.7 trillion parameters, backprop takes ~2 minutes per batch.")


> **What just happened?** The forward pass computed the output. The backward pass used the chain rule to trace the error from output back through each layer, computing how much each weight contributed. Both weights adjusted simultaneously. After 50 epochs: output ≈ 8.0 (target). In real LLMs, PyTorch's autograd system does this automatically for billions of parameters. You'll never write backprop by hand — but understanding the chain rule helps you debug exploding gradients (Module 7) and choose learning rates.


### Step 7 · Perplexity & Activation Functions — Evaluate + Power Up

Perplexity measures how good your LLM is. Activation functions give neural networks their power.

**`perplexity_activation.py`** · _Block 7 of 7_

PERPLEXITY — How good is your LLM? — ACTIVATION FUNCTIONS — Why neural networks can learn anything


In [ ]:
# =============================================
# PERPLEXITY — How good is your LLM?
# ACTIVATION FUNCTIONS — Why neural networks can learn anything
# =============================================

import numpy as np

# ── PERPLEXITY = exp(cross-entropy) ──
# Lower = better. Measures "how confused" the model is.

def cross_entropy(probs, target):
    return -np.log(probs[target] + 1e-10)

def perplexity(ce_loss):
    return np.exp(ce_loss)

# Simulate different model qualities
scenarios = [
    ("Random guess (50K vocab)",  [1/50000]*50000, 0),
    ("Bad model",                [0.01]*100, 0),
    ("Average model",            [0.15, 0.1, 0.08]+[0.01]*67, 0),
    ("Good model (GPT-3)",       [0.6, 0.15, 0.1]+[0.005]*30, 0),
    ("Excellent model (GPT-4)",  [0.85, 0.08, 0.04]+[0.001]*30, 0),
]

print("PERPLEXITY — How confused is the model?\n")
print(f"{'Model':<30s} {'P(correct)':>10s} {'CE Loss':>8s} {'Perplexity':>11s}")
print("-" * 65)
for name, probs, target in scenarios:
    probs = np.array(probs)
    probs = probs / probs.sum()
    ce = cross_entropy(probs, target)
    ppl = perplexity(ce)
    print(f"  {name:<28s} {probs[target]:>9.4f} {ce:>8.3f} {ppl:>10.1f}")

# ── ACTIVATION FUNCTIONS ──
# Without these, 96 layers = 1 layer (just a linear function)

print(f"\n\nACTIVATION FUNCTIONS — Why depth matters\n")

x = np.linspace(-3, 3, 7)

# ReLU: max(0, x) — simple but effective
relu = np.maximum(0, x)

# GELU: x × Φ(x) — used in GPT, BERT, Gemini (smoother than ReLU)
gelu = x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

print(f"{'x':>6s} {'ReLU':>8s} {'GELU':>8s}  Note")
print("-" * 45)
for i in range(len(x)):
    note = ""
    if x[i] < 0: note = "GELU lets small negatives through"
    if x[i] == 0: note = "Both zero at origin"
    print(f"  {x[i]:>5.1f} {relu[i]:>8.3f} {gelu[i]:>8.3f}  {note}")

print(f"\nKey insight: GELU is smoother → better gradients → faster training")
print(f"GPT-4, BERT, Gemini all use GELU. ReLU is used in older models.")


> **What just happened?** Perplexity: A perplexity of 50,000 means the model is randomly guessing. GPT-4's perplexity of ~6 means it narrows down to about 6 candidates per word — that's why it sounds fluent. When fine-tuning (Module 7), watch perplexity drop → that's your model improving. Activation functions: Without them, a 96-layer network equals a 1-layer network (matrix math collapses). ReLU and GELU add nonlinearity — the ability to learn curves, not just straight lines. Every transformer layer has: attention → add+normalize → GELU → add+normalize.


---

## Wrap-up

You've walked through all 7 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
